<h1 align="center">TÉCNICO EM CIÊNCIA DE DADOS</h1>
<h2 align="center">Roteiro de Atividade Prática</h2>

**Componente:** Análise Exploratória de Dados e Inteligência de Negócios  
**Unidade Curricular:** Projeto Análise Exploratória de Dados  
**Tema da Semana:** Projeto AED – Desenvolvimento das Análises  
**Semana 13 — 3º bimestre**  
**Aula 1:** Desenvolvimento das análises do projeto

**Nome(s):**  
**Turma:**

# Do aproveitamento ao resultado comercial

**Situação profissional**

Vocês integram a equipe de análise de dados de uma rede brasileira de oito papelarias. A investigação inicial indicou a redução da espera como uma possível prioridade de melhoria. Antes de decidir onde investir, a direção quer saber se as condições associadas ao aproveitamento comercial também estão relacionadas ao resultado das lojas após as despesas.

**Pergunta investigativa:** Quais condições estão associadas ao desempenho comercial das lojas e qual aspecto merece prioridade para uma primeira ação de melhoria?

**Missão:** Aprofundem a investigação e produzam novas evidências para auxiliar a decisão da direção.

Período: 6 de outubro a 30 de novembro de 2025. Cada registro representa uma loja em um dia de operação. Os dados são simulados para fins didáticos. Tempo total: 43 minutos. Registrem as respostas Q1 a Q5 no notebook.

Organizem-se em duplas no laboratório de informática, com um computador por dupla. Acompanhem as orientações projetadas e mantenham o espaço de circulação do professor livre.

Alternem os papéis: um integrante executa as células e o outro confere as saídas e participa do registro das respostas.

## Objetivos

- Aplicar técnicas de análise exploratória adequadas à pergunta investigativa e às características dos dados do projeto.
- Construir indicadores e comparar o desempenho comercial das lojas com os mesmos critérios de recorte.
- Examinar relações entre condições operacionais e desempenho e registrar evidências para o aprofundamento da investigação.

## 1. Preparem os dados — 5 min

Deixem o notebook e o CSV na mesma pasta. Abram o notebook, selecionem um kernel Python disponível e executem as células na ordem, utilizando o código preparado. Não é necessário recuperar arquivos produzidos em outras aulas.

**Arquivo de dados:** `rede_papelarias_B2_integrado.csv`. Bibliotecas: pandas, pathlib e IPython.display. No Google Colab, enviem o CSV para a sessão antes de executar.

Confiram a retirada da duplicação integral, a padronização das informações monetárias, a seleção do período e a disponibilidade dos dados necessários às análises. Preservem os valores ausentes; não os substituam por zero. Utilizem, em cada análise, os registros que possuem as informações necessárias.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

arquivo = Path("rede_papelarias_B2_integrado.csv")
if not arquivo.is_file():
    raise FileNotFoundError("Deixem o CSV indicado na mesma pasta do notebook. No Colab, enviem o CSV para a sessão.")

dados = pd.read_csv(arquivo, encoding="utf-8-sig")
print("Arquivo:", arquivo.name)
print("Registros originais:", len(dados))
print("Variáveis:", dados.shape[1])
print("Cópias integrais:", dados.duplicated().sum())
dados_trabalho = dados.drop_duplicates().copy()
print("Registros após retirar cópias integrais:", len(dados_trabalho))
if dados_trabalho["id_registro"].duplicated().any():
    raise ValueError("Há identificadores repetidos após retirar cópias integrais. Consultem o professor.")

<details>
<summary>Dicionário de dados — consulte quando necessário</summary>

| Coluna | Informação |
|---|---|
| `id_registro` | Identificador do registro. |
| `data` | Data da operação diária. |
| `loja` | Identificador da loja. |
| `entradas` | Número de entradas no dia; não identifica pessoas distintas. |
| `horas_equipe` | Soma das horas trabalhadas pela equipe no dia. |
| `investimento_divulgacao` | Valor investido em divulgação no dia (R$). |
| `canal_divulgacao` | Canal principal de divulgação no dia. |
| `desconto_campanha_pct` | Percentual de desconto sobre produtos participantes da campanha. |
| `disponibilidade_produtos_pct` | Percentual dos produtos monitorados disponíveis na abertura da loja. |
| `espera_media_min` | Tempo médio até o início do atendimento, em minutos. |
| `cupons_emitidos` | Quantidade de transações registradas no dia. |
| `receita_liquida` | Receita após descontos comerciais (R$). |
| `lucro_bruto` | Receita líquida menos custo das mercadorias vendidas (R$). |
| `despesas_com_pessoal` | Despesas com pessoal registradas no dia (R$). |
| `outras_despesas_operacionais` | Outras despesas operacionais registradas no dia (R$). |

</details>

In [ ]:
# Padronizar somente as duas colunas de despesas, sem preencher ausências.
despesas = ["despesas_com_pessoal", "outras_despesas_operacionais"]
for coluna in despesas:
    original = dados_trabalho[coluna]
    texto = original.astype("string").str.strip().str.replace("R$", "", regex=False).str.strip()
    texto = texto.str.replace(",", ".", regex=False)
    convertido = pd.to_numeric(texto, errors="raise")
    assert convertido.isna().sum() == original.isna().sum()
    dados_trabalho[coluna] = convertido.astype(float)

dados_trabalho["data"] = pd.to_datetime(dados_trabalho["data"], errors="raise")
periodo = dados_trabalho.loc[dados_trabalho["data"].between("2025-10-06", "2025-11-30")].copy()
print("Registros no período:", len(periodo))
display(periodo.groupby("loja").size().to_frame("Loja-dias"))
print("Ausências nas variáveis necessárias:")
display(periodo[["espera_media_min", *despesas]].isna().sum().to_frame("Ausências"))
print("Registros com espera conhecida:", periodo["espera_media_min"].notna().sum())

## 2. Construam os indicadores — 7 min

Recuperem o indicador de aproveitamento comercial: (cupons emitidos ÷ entradas) × 100. Esse indicador expressa cupons por 100 entradas, não a proporção de pessoas distintas que compraram.

Calculem o resultado após as despesas registradas: lucro bruto − investimento em divulgação − despesas com pessoal − outras despesas operacionais. A medida não equivale a lucro líquido.

In [ ]:
# Recuperar o aproveitamento e calcular o resultado após as despesas registradas.
if periodo["entradas"].isna().any() or (periodo["entradas"] <= 0).any():
    raise ValueError("Há denominadores ausentes ou não positivos. Consultem o professor.")
periodo["cupons_por_100_entradas"] = 100 * periodo["cupons_emitidos"] / periodo["entradas"]
periodo["resultado_apos_despesas"] = (
    periodo["lucro_bruto"] - periodo["investimento_divulgacao"]
    - periodo["despesas_com_pessoal"] - periodo["outras_despesas_operacionais"]
)
print("Registros com resultado calculável:", periodo["resultado_apos_despesas"].notna().sum())
display(periodo[["id_registro", "loja", "entradas", "cupons_emitidos", "cupons_por_100_entradas", "lucro_bruto", "resultado_apos_despesas"]].head().round(2))

**Q1.** Qual dos dois indicadores financeiros — lucro bruto ou resultado após as despesas registradas — é mais adequado para examinar o desempenho das lojas considerando as novas informações? Justifiquem.

**Resposta:**

*Registrem aqui a resposta da dupla.*

## 3. Comparem os resultados comerciais — 9 min

Construam uma tabela com o lucro bruto médio diário e o resultado médio diário após as despesas registradas de cada loja. Utilizem os mesmos registros para calcular os dois indicadores.

In [ ]:
financeiro = periodo.dropna(subset=["lucro_bruto", "resultado_apos_despesas"])
comparacao_financeira = financeiro.groupby("loja").agg(
    n=("id_registro", "size"),
    lucro_bruto_medio=("lucro_bruto", "mean"),
    resultado_medio=("resultado_apos_despesas", "mean")
)
print("Registros usados nos dois indicadores:", len(financeiro))
display(comparacao_financeira.rename(columns={"n": "Observações", "lucro_bruto_medio": "Lucro bruto médio (R$)", "resultado_medio": "Resultado médio após despesas (R$)"}).round(2))

**Q2.** Identifiquem as lojas que apresentam lucro bruto médio positivo, mas resultado médio negativo após as despesas registradas. Registrem os respectivos valores.

**Resposta:**

*Registrem aqui a resposta da dupla.*

## 4. Examinem as relações entre condições operacionais e desempenho — 15 min

Calculem as correlações de Pearson entre tempo médio de espera, horas de equipe e disponibilidade de produtos e os indicadores cupons por 100 entradas, receita líquida e resultado após as despesas registradas.

Utilizem todos os registros completos do período e depois apenas aqueles da faixa de 150 a 199 entradas. Em cada recorte, usem o mesmo conjunto de registros nas nove correlações.

Lembrem-se: a intensidade da associação é examinada pelo valor absoluto do coeficiente; o sinal indica a direção da associação.

In [ ]:
condicoes = ["espera_media_min", "horas_equipe", "disponibilidade_produtos_pct"]
indicadores = ["cupons_por_100_entradas", "receita_liquida", "resultado_apos_despesas"]
variaveis = condicoes + indicadores

# Mesmos registros para todas as nove correlações de cada recorte.
completos = periodo.dropna(subset=variaveis).copy()
faixa = completos.loc[completos["entradas"].between(150, 199)].copy()

rotulos = {"espera_media_min": "Espera média (min)", "horas_equipe": "Horas de equipe (h)",
           "disponibilidade_produtos_pct": "Disponibilidade (%)", "cupons_por_100_entradas": "Cupons/100 entradas",
           "receita_liquida": "Receita líquida (R$)", "resultado_apos_despesas": "Resultado após despesas (R$)"}

def correlacoes(d):
    return d[variaveis].corr(method="pearson").loc[condicoes, indicadores]

corr_periodo = correlacoes(completos)
corr_faixa = correlacoes(faixa)
print("Período completo — observações:", len(completos))
display(corr_periodo.rename(index=rotulos, columns=rotulos).round(3))
print("Faixa de 150 a 199 entradas — observações:", len(faixa))
display(corr_faixa.rename(index=rotulos, columns=rotulos).round(3))

**Q3.** Identifiquem a condição operacional que apresenta a associação linear de maior intensidade com cada indicador de desempenho, nos dois recortes. Comparem os resultados obtidos.

**Resposta:**

*Registrem aqui a resposta da dupla.*

Utilizem os mesmos registros completos da faixa na tabela comparativa por loja.

In [ ]:
# Comparar as oito lojas utilizando os mesmos casos completos da faixa.
comparacao_lojas = faixa.groupby("loja")[variaveis].mean()
comparacao_lojas.insert(0, "n", faixa.groupby("loja").size())
comparacao_lojas = comparacao_lojas.reindex(sorted(periodo["loja"].unique()))
display(comparacao_lojas.rename(columns={"n": "Observações", **rotulos}).round(2))

**Q4.** Construam uma tabela comparativa das oito lojas na faixa de 150 a 199 entradas, apresentando as médias das condições operacionais, dos indicadores de desempenho e a quantidade de observações por loja. Comparem especialmente as lojas L3 e L4. Quais diferenças de desempenho não são suficientemente esclarecidas pelos indicadores de espera e disponibilidade geral?

**Resposta:**

*Registrem aqui a resposta da dupla.*

## 5. Registrem as evidências e encaminhem a investigação — 7 min

**Q5.** Com base nas comparações realizadas, indiquem uma informação que o indicador geral de disponibilidade de produtos não fornece e que poderia ajudar a compreender as diferenças de desempenho comercial entre as lojas. Justifiquem a solicitação dessa informação à direção.

**Resposta:**

*Registrem aqui a resposta da dupla.*

Nos minutos finais, discutam com o professor as solicitações propostas e confiram se elas são sustentadas pelas evidências obtidas. Associação não demonstra causalidade nem garante o resultado de uma intervenção.

## Critérios para avaliação da atividade

- Aplica os procedimentos de preparação e constrói os indicadores com os critérios definidos.
- Compara as lojas utilizando os mesmos registros para os indicadores financeiros e os recortes especificados.
- Determina e compara as associações lineares, considerando o sinal, a intensidade e a quantidade de observações.
- Identifica diferenças de desempenho que os indicadores disponíveis não esclarecem e justifica a solicitação de informação adicional à direção.
- Registra as evidências e respeita os limites da análise, sem afirmar causalidade.

**Registro da atividade: notebook preenchido e salvo por dupla ao final da aula. Nenhum arquivo produzido pelo estudante será exigido como entrada da próxima aula.**

Os resultados descrevem somente o conjunto simulado para fins didáticos.